# 🚀 Tái Hiện Thuật Toán LiDAR (Lookahead Sample Reward Guidance) - Bảng 2
### **Bài báo**: [Lookahead Sample Reward Guidance for Test-Time Scaling of Diffusion Models (ICML 2026 Spotlight)](https://arxiv.org/abs/2602.03211)
### **Mục tiêu**: Chạy lại mã nguồn và tái hiện kết quả của **Bảng 2**: **SD v1.5 + LiDAR (DPM-5 / $n=50$)** trên tập prompt GenEval.

---
### 📊 Kết quả mục tiêu trong bài báo (Bảng 2):
| Mô hình Backbone | Phương pháp Sampling | ImageReward (↑) | CLIP Score (↑) | HPS v2.1 (↑) | GenEval (↑) | Thời gian (s/lần) | VRAM (GiB) |
| :--- | :--- | :---: | :---: | :---: | :---: | :---: | :---: |
| **SD v1.5 (DDPM 100 bước)** | **LiDAR (DPM-5 / $n=50$)** | **0.384** | **0.278** | **0.276** | **0.478** | 13.41s | 8.90 GiB |
| **SD v1.5 (DDIM 50 bước)** | **LiDAR (DPM-5 / $n=50$)** | **0.378** | **0.278** | **0.277** | **0.475** | 9.92s | 8.90 GiB |
| *Vanilla SD v1.5 (DDPM 100)* | Baseline gốc | 0.001 | 0.271 | 0.263 | 0.426 | 7.07s | 8.90 GiB |
| *Vanilla SD v1.5 (DDIM 50)* | Baseline gốc | -0.125 | 0.269 | 0.270 | 0.423 | 3.58s | 8.90 GiB |

---
### 🛠 Cơ Chế Lưu Trữ & Chạy Tiếp Tục (Resume Checkpoint):
1. **Lưu tự động sau mỗi Prompt**: Mỗi prompt khi sinh xong sẽ lập tức lưu latent `latent.pt` và chỉ số `results.json` vào thư mục riêng (`00000/`, `00001/`,...).
2. **Tự động bỏ qua prompt đã hoàn thành**: Nếu phiên làm việc bị ngắt kết nối hoặc bạn tắt đi bật lại, mã nguồn có cờ `--resume` sẽ quét thư mục, tự động nạp kết quả cũ và **chỉ chạy tiếp các prompt còn lại**.
3. **Chuyển giao giữa các phiên Kaggle mới**: Bạn chỉ cần tải file `.zip` kết quả của phiên trước về. Khi mở phiên mới, giải nén file zip vào là code sẽ chạy nối tiếp mượt mà!


## 1. Kiểm tra Môi trường Hệ thống & GPU


In [ ]:
import os
import sys
import torch

print(f"Phiên bản Python: {sys.version}")
print(f"Phiên bản PyTorch: {torch.__version__}")
print(f"Hỗ trợ CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Tên GPU: {torch.cuda.get_device_name(0)}")
    print(f"Bộ nhớ VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
!nvidia-smi


## 2. Thiết lập Mã Nguồn & Cài đặt Thư viện


In [ ]:
# Thiết lập thư mục làm việc trên Kaggle
import os

REPO_DIR = "/kaggle/working/RS-LiDAR"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/leekwanreal/RS-LiDAR.git {REPO_DIR}

if os.path.exists(f"{REPO_DIR}/Diffusion-LiDAR-Sampling"):
    WORKDIR = f"{REPO_DIR}/Diffusion-LiDAR-Sampling"
else:
    WORKDIR = REPO_DIR

os.chdir(WORKDIR)
%cd {WORKDIR}
print("Thư mục làm việc hiện tại:", os.getcwd())

# Cài đặt chính xác các phiên bản theo requirements.txt của tác giả
!pip install -q -r requirements.txt
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q transformers==4.38.2 accelerate==1.2.1


## 2.1. [TÙY CHỌN] Khôi Phục Checkpoint Từ Phiên Trước Để Chạy Tiếp
Nếu ở phiên trước bạn đã chạy dở một phần và đã tải file zip về, bạn có thể upload file zip lên Kaggle (như 1 Dataset) hoặc đặt vào `/kaggle/working` rồi chạy ô này để giải nén tiếp tục.


In [ ]:
# Ô này dùng để giải nén checkpoint cũ (nếu bạn bắt đầu phiên Kaggle mới)
import glob
import os

os.chdir(WORKDIR)

# Tìm các file zip cũ nếu có upload vào Kaggle
old_lookahead_zips = glob.glob("/kaggle/input/**/lookahead_*.zip", recursive=True) + glob.glob("/kaggle/working/lookahead_*.zip")
old_target_zips = glob.glob("/kaggle/input/**/*_results.zip", recursive=True) + glob.glob("/kaggle/working/*_results.zip")

for z in old_lookahead_zips:
    print(f"Đang khôi phục Lookahead checkpoint từ: {z}")
    !unzip -q -o {z} -d {WORKDIR}/

for z in old_target_zips:
    print(f"Đang khôi phục Target checkpoint từ: {z}")
    !unzip -q -o {z} -d {WORKDIR}/

print("✅ Đã kiểm tra và sẵn sàng dữ liệu checkpoint.")


## 3. Cấu hình Siêu Tham Số Thí Nghiệm (Thiết lập theo Bảng 2)
Các tham số chuẩn được trích xuất trực tiếp từ bài báo (Mục 4.1 & Phụ lục B.1):
- **Phase 1 (Lookahead Sampler)**: Sử dụng DPM-Solver với $S=5$ bước (`--num_inference_steps=5`), tạo $n=50$ hạt/mẫu (`--num_particles=50`).
- **Phase 2 (LiDAR Target Sampler)**: SD v1.5 với 100 bước DDPM (`--num_inference_steps=100`, `--eta=1.0`) hoặc 50 bước DDIM (`--num_inference_steps=50`, `--eta=0.0`).
- **Tham số Guidance**: Hệ số scale $s=12.5$ (`--scale=12.5`), nhiệt độ $\lambda=5000$ (`--lmbda=5000`), ngưỡng dừng sớm $T_{end}=200$ (`--resample_t_end=200`, tương ứng khoảng denoising $[1.0, 0.2]$).
- **Số ảnh đánh giá**: $N=4$ ảnh cho mỗi prompt theo chuẩn giao thức benchmark GenEval.


In [ ]:
# ==================== CẤU HÌNH SIÊU THAM SỐ ====================
SEED = 100                       # Random seed (100 hoặc 42)
NUM_LOOKAHEAD_PARTICLES = 50     # Số hạt lookahead n = 50
LOOKAHEAD_STEPS = 5              # Số bước DPM-Solver = 5 (DPM-5)
LOOKAHEAD_TAG = f"{SEED}_{NUM_LOOKAHEAD_PARTICLES}_{LOOKAHEAD_STEPS}"

# Tham số lấy mẫu đích Phase 2 (SD v1.5 với DDPM 100 bước)
MODEL_NAME = "runwayml/stable-diffusion-v1-5"
NUM_TARGET_STEPS = 100           # 100 bước cho DDPM (hoặc 50 bước cho DDIM)
ETA = 1.0                        # eta = 1.0 cho DDPM (hoặc 0.0 cho DDIM)
TARGET_PARTICLES = 4             # 4 ảnh trên mỗi prompt (chuẩn đánh giá GenEval)
SCALE = 12.5                     # Hệ số guidance s = 12.5 cho SD v1.5
LAMBDA = 5000                    # Hệ số nhiệt độ lambda = 5000
RESAMPLE_T_END = 200             # Ngưỡng kết thúc guidance sớm [1.0, 0.2]
TOP_K = 50                       # Chọn top-k lookaheads (50)

# Dữ liệu Prompt và Giới hạn số lượng
PROMPT_FILE = "prompt_files/geneval_metadata.jsonl"
# Đặt MAX_PROMPTS = 553 để chạy toàn bộ benchmark, hoặc chỉnh thành 10-20 để kiểm tra nhanh
MAX_PROMPTS = 553

RUN_NAME = f"LiDAR_SD15_DPM5_n50_steps{NUM_TARGET_STEPS}_seed{SEED}"

print(f"Tên lượt chạy: {RUN_NAME}")
print(f"Đường dẫn Lookahead: {LOOKAHEAD_TAG}")
print(f"Tổng số Prompt cần xử lý: {MAX_PROMPTS}")


## 4. Giai Đoạn 1 (Phase 1): Lấy Mẫu Lookahead & Đánh Giá Reward
Giai đoạn này sinh ra $n=50$ hạt cho mỗi prompt bằng 5 bước giải DPM-Solver cực nhanh, sau đó chấm điểm bằng `ImageReward` (và `Clip-Score`).
- **Cơ chế tự động tiếp tục (`--resume`)**: Nếu bị ngắt giữa chừng, khi chạy lại cell này mã nguồn sẽ tự động kiểm tra `results.json` của từng prompt và **bỏ qua ngay lập tức các prompt đã xong**!


In [ ]:
# Thực thi Giai đoạn 1: Lookahead Sampling
import os
os.chdir(WORKDIR)

lookahead_cmd = f"""cd {WORKDIR} && python lookahead_sampling.py \
    --seed={SEED} \
    --num_particles={NUM_LOOKAHEAD_PARTICLES} \
    --num_inference_steps={LOOKAHEAD_STEPS} \
    --model_name="{MODEL_NAME}" \
    --prompt_path="{PROMPT_FILE}" \
    --max_prompt={MAX_PROMPTS} \
    --guidance_reward_fn="ImageReward" \
    --metrics_to_compute="ImageReward#Clip-Score" \
    --resume
"""

print(">>> Đang bắt đầu Giai đoạn 1 (Lookahead Sampling)...")
!{lookahead_cmd}


## 5. Giai Đoạn 2 (Phase 2): Lấy Mẫu Đích LiDAR Sampling
Giai đoạn này thực hiện quá trình lấy mẫu khuếch tán với hướng dẫn LiDAR (Lookahead Sample Reward Guidance) hướng về các mẫu lookahead có reward cao.
- **Cơ chế tự động tiếp tục (`--resume`)**: Tự động bỏ qua các prompt đã chạy xong và tiếp tục prompt tiếp theo.


In [ ]:
# Thực thi Giai đoạn 2: LiDAR Steering Sampling
import os
os.chdir(WORKDIR)

lidar_cmd = f"""cd {WORKDIR} && python LiDAR_sampling.py \
    --seed={SEED} \
    --model_name="{MODEL_NAME}" \
    --num_particles={TARGET_PARTICLES} \
    --num_inference_steps={NUM_TARGET_STEPS} \
    --eta={ETA} \
    --use_rag \
    --lookahead_path="{LOOKAHEAD_TAG}" \
    --top_k={TOP_K} \
    --scale={SCALE} \
    --lmbda={LAMBDA} \
    --resample_t_end={RESAMPLE_T_END} \
    --prompt_path="{PROMPT_FILE}" \
    --max_prompt={MAX_PROMPTS} \
    --guidance_reward_fn="ImageReward" \
    --metrics_to_compute="ImageReward#Clip-Score#Clip-Diversity#HumanPreference#AS" \
    --save_individual_images \
    --run_name="{RUN_NAME}" \
    --resume
"""

print(">>> Đang bắt đầu Giai đoạn 2 (LiDAR Sampling)...")
!{lidar_cmd}


## 6. Đánh Giá Định Lượng & So Sánh với Bảng 2
Đọc kết quả từ file `final_metrics.json` và lập bảng so sánh trực tiếp kết quả chạy thực tế với kết quả công bố trong Bảng 2 của bài báo.


In [ ]:
import json
import pandas as pd
from IPython.display import display

results_file = f"{WORKDIR}/Target_samples/{RUN_NAME}/final_metrics.json"

if os.path.exists(results_file):
    with open(results_file, "r") as f:
        metrics = json.load(f)
    
    reproduced_metrics = {
        "ImageReward (IR)": f"{metrics.get('ImageReward', {}).get('mean', 0.0):.4f}",
        "CLIP Score": f"{metrics.get('Clip-Score', {}).get('mean', 0.0):.4f}",
        "HPS v2.1": f"{metrics.get('HumanPreference', {}).get('mean', 0.0):.4f}",
        "CLIP Diversity": f"{metrics.get('Clip-Diversity', {}).get('mean', 0.0):.4f}",
        "Aesthetic Score (AS)": f"{metrics.get('AS', {}).get('mean', 0.0):.4f}",
    }
    
    paper_target_ddpm = {
        "ImageReward (IR)": "0.384",
        "CLIP Score": "0.278",
        "HPS v2.1": "0.276",
        "CLIP Diversity": "-",
        "Aesthetic Score (AS)": "-"
    }

    paper_target_ddim = {
        "ImageReward (IR)": "0.378",
        "CLIP Score": "0.278",
        "HPS v2.1": "0.277",
        "CLIP Diversity": "-",
        "Aesthetic Score (AS)": "-"
    }

    df = pd.DataFrame([
        reproduced_metrics,
        paper_target_ddpm if NUM_TARGET_STEPS == 100 else paper_target_ddim
    ], index=["Kết quả Chạy Thực Tế", f"Bài Báo Bảng 2 (DD{'PM-100' if NUM_TARGET_STEPS==100 else 'IM-50'})"])
    
    print("
=================== 📊 BẢNG SO SÁNH KẾT QUẢ ĐỊNH LƯỢNG ===================")
    display(df)
else:
    print(f"Chưa tìm thấy file kết quả tại {results_file}. Vui lòng chạy Giai đoạn 2 trước.")


## 7. Trực Quan Hóa Các Ảnh Mẫu Đã Sinh
Hiển thị lưới hình ảnh 4 ảnh/prompt để kiểm tra trực quan chất lượng sinh ảnh.


In [ ]:
import glob
from PIL import Image
import matplotlib.pyplot as plt

target_dir = f"{WORKDIR}/Target_samples/{RUN_NAME}"
grid_images = sorted(glob.glob(f"{target_dir}/*/grid.png"))

if grid_images:
    print(f"Tìm thấy {len(grid_images)} lưới ảnh prompt. Đang hiển thị 3 prompt đầu tiên:")
    for img_path in grid_images[:3]:
        img = Image.open(img_path)
        plt.figure(figsize=(16, 4))
        plt.imshow(img)
        plt.axis('off')
        plt.title(f"Chỉ số Prompt: {os.path.basename(os.path.dirname(img_path))}")
        plt.show()
else:
    print("Chưa tìm thấy ảnh lưới mẫu nào.")


## 8. Đóng Gói & Xuất Kết Quả (Tải về Máy với 1 Cú Click)
Nén các thư mục kết quả ảnh và lookahead thành file `.zip` lưu trong `/kaggle/working` để bạn có thể tải về máy dễ dàng.


In [ ]:
output_zip_target = f"/kaggle/working/{RUN_NAME}_results.zip"
output_zip_lookahead = f"/kaggle/working/lookahead_{LOOKAHEAD_TAG}.zip"

if os.path.exists(f"{WORKDIR}/Target_samples/{RUN_NAME}"):
    !zip -q -r {output_zip_target} {WORKDIR}/Target_samples/{RUN_NAME}
    print(f"✅ Đã nén kết quả Target: {output_zip_target} ({os.path.getsize(output_zip_target) / (1024*1024):.2f} MB)")

if os.path.exists(f"{WORKDIR}/Lookahead_samples/{LOOKAHEAD_TAG}"):
    !zip -q -r {output_zip_lookahead} {WORKDIR}/Lookahead_samples/{LOOKAHEAD_TAG}
    print(f"✅ Đã nén kết quả Lookahead: {output_zip_lookahead} ({os.path.getsize(output_zip_lookahead) / (1024*1024):.2f} MB)")

print("
🎉 Hoàn tất đóng gói! Bạn có thể tải các file zip này về máy từ mục Output trên Kaggle.")
